In [ ]:
import os
import pandas as pd
import csv


"""
if()
if timestamps["time"].max() >= target:
    candidates.append(timestamps[timestamps["time"] >= target].loc[timestamps["time"].idxmin()])


index = bisect_left(
    timestamps,
    target
)
    

if index < len(timestamps):
    candidates.append(timestamps[index])

if index > 0:
    candidates.append(timestamps[index - 1])


if not candidates:
    return None

nearest = min(
    candidates["time"],
    key=lambda value: abs(value - target)
)

difference = abs(
    nearest - target
)


if difference <= SYNC_TOLERANCE_NS:
    return nearest

return None
"""

def create_synchronization_csv():

    print("\nCreating synchronization.csv...")

    #OUTPUT_DIR = os.path.join("C", "Users", "stratklar", "Documents", "Markus", "ASA", "Indien", "mobile_data", "sample1")
    OUTPUT_DIR = r"C:\Users\stratklar\Documents\Markus\ASA\Indien\mobile_data\sample3"
    accelerometer = pd.read_csv(os.path.join(OUTPUT_DIR, "Accelerometer.csv"))
    gravity = pd.read_csv(os.path.join(OUTPUT_DIR, "Gravity.csv"))  
    locationcsv = pd.read_csv(os.path.join(OUTPUT_DIR, "Location.csv"))
    gyroscope = pd.read_csv(os.path.join(OUTPUT_DIR, "Gyroscope.csv"))
    orientation = pd.read_csv(os.path.join(OUTPUT_DIR, "Orientation.csv"))

    def nearest_timestamp(string, target):

        #if not timestamps:
        #    return None
        #candidate = None
        index = -1
        SYNC_TOLERANCE_NS = 1000000000

        if string == "accelerometer":
            return len(accelerometer[accelerometer["time"] < target])

        elif string == "gravity":
            timestamps = gravity 

        elif string == "location":
            timestamps = locationcsv

        elif string == "gyroscope":
            timestamps = gyroscope

        elif string == "orientation":
            timestamps = orientation

        else:
            return None

        if timestamps["time"].min() <= target:
            history = timestamps[timestamps["time"] <= target]
            #candidate = history.loc[timestamps["time"].idxmax()]
            if len(history) > 0:
                index = len(history) - 1
            else:
                return 0
            #index = timestamps.index.get_loc(index)

        else:
            return 0
        
        if len(history) < len(timestamps):
            if timestamps.iloc[index + 1]["time"] - target < target - timestamps.iloc[index]["time"]:
                index += 1

        if abs(target - timestamps.iloc[index]["time"]) < SYNC_TOLERANCE_NS:
            return index
        print("none")
        return None
    
    reference_timestamps = accelerometer["time"]

    sync_path = os.path.join(
        OUTPUT_DIR,
        "synchronization.csv"
    )

    sync_path2 = os.path.join(
        OUTPUT_DIR,
        "synchronization2.csv"
    )

    with open(
        sync_path,
        "w",
        newline="",
        encoding="utf-8"
    ) as file_handle:

        writer = csv.writer(file_handle)

        writer.writerow([
            "common_timestamp_ns",
            "common_timestamp_seconds",

            "accelerometer_timestamp_ns",
            "gravity_timestamp_ns",
            "location_timestamp_ns",
            "gyroscope_timestamp_ns",
            "orientation_timestamp_ns"
        ])

        #reference_timestamps = sensor_timestamps[
        #   reference_topic
        #]

        synchronized_count = 0
        count = 0
        rows = []

        for common_timestamp in reference_timestamps:

            count += 1
            """
            def find_timestamp(topic):
                return nearest_timestamp(
                    topic,
                    common_timestamp
                )
            gravity_timestamp = nearest_timestamp(gravity["time"].dropna().tolist(), common_timestamp)
            location_timestamp = nearest_timestamp(location["time"].dropna().tolist(), common_timestamp)
            gyroscope_timestamp = nearest_timestamp(gyroscope["time"].dropna().tolist(), common_timestamp)
            orientation_timestamp = nearest_timestamp(orientation["time"].dropna().tolist(), common_timestamp)
            """
            accelerometer_index = nearest_timestamp("accelerometer", common_timestamp)
            gravity_index = nearest_timestamp("gravity", common_timestamp)
            location_index = nearest_timestamp("location", common_timestamp)
            gyroscope_index = nearest_timestamp("gyroscope", common_timestamp)
            orientation_index = nearest_timestamp("orientation", common_timestamp)
            #lidar_timestamp = find_timestamp("/scan")

            #print(type(gravity))
            #print(gravity.shape)
            #print(type(gravity_index))
            #print(gravity_index)
            #print(gravity.head())
            # Require all three main sensors and all cameras.
            # CameraInfo is optional because it can be published
            # at a much lower frequency.
            if (accelerometer_index is None
                or gravity_index is None
                or location_index is None
                or gyroscope_index is None
                or orientation_index is None
            ):
                
                continue
            """
            writer.writerow([
                common_timestamp,
                common_timestamp / 1_000_000_000,
                ] +
                accelerometer.iloc[accelerometer_index].tolist() +
                #gravity.iloc[gravity_index].tolist() +
                locationcsv.iloc[location_index].tolist() + 
                gyroscope.iloc[gyroscope_index].tolist() +
                orientation.iloc[orientation_index].tolist()
            )
            """

            row = (
                [common_timestamp, common_timestamp / 1_000_000_000]
                + accelerometer.iloc[accelerometer_index].tolist()
                + gravity.iloc[gravity_index].tolist()
                + locationcsv.iloc[location_index].tolist()
                + gyroscope.iloc[gyroscope_index].tolist()
                + orientation.iloc[orientation_index].tolist()
            )
            rows.append(row)

            synchronized_count += 1

        columns = (
            ["timestamp", "timestamp_seconds"]
            + ["accelerometer_" + c for c in accelerometer.columns]
            + ["gravity_" + c for c in gravity.columns]
            + ["location_" + c for c in locationcsv.columns]
            + ["gyroscope_" + c for c in gyroscope.columns]
            + ["orientation_" + c for c in orientation.columns]
        )

        merged_df = pd.DataFrame(rows, columns=columns)
    merged_df.to_csv(sync_path2, index=False)
    print(
        f"Synchronized samples: {synchronized_count}"
    )
    print(count)

print("synchronization successful")
create_synchronization_csv()

synchronization successful

Creating synchronization.csv...
Synchronized samples: 6099
6099
